# ECHO-17 · Real UCI Hydraulic Sensor ML (Colab GPU)
Run this notebook in **Runtime → Change runtime type → T4 GPU**. This notebook installs dependencies, downloads UCI experimental data, performs CPU baseline and optional TabPFN-3.5 inference. There is **no synthetic demo data**. Check and accept TabPFN model terms before starting.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select the latest echo-17 project ZIP
import zipfile, io, os
name = next(k for k in uploaded if k.endswith(".zip"))
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
    z.extractall("/content")
print("Repo extracted")

In [ ]:
%cd /content/echo-17/backend
!pip -q install -r requirements.txt tabpfn

In [ ]:
!python -m ml.download_data
!python -m ml.prepare
!python -m ml.train_baseline

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
!python -m ml.train_tabpfn

In [ ]:
import json
from pathlib import Path
for name in ["baseline_metrics.json", "tabpfn_metrics.json"]:
    print(name, json.loads((Path("artifacts")/name).read_text()))

In [ ]:
import zipfile
from google.colab import files
from pathlib import Path
with zipfile.ZipFile("/content/echo17_results.zip","w") as z:
    for p in Path("artifacts").glob("*.csv"): z.write(p, "artifacts/"+p.name)
    for p in Path("artifacts").glob("*.json"): z.write(p,"artifacts/"+p.name)
    for p in Path("data/processed").glob("*.csv"): z.write(p,"data/processed/"+p.name)
    for p in Path("data/processed").glob("*.npz"): z.write(p,"data/processed/"+p.name)
files.download("/content/echo17_results.zip")